In [148]:
import pandas as pd

df_vendas = pd.read_csv("datasets/vendas_2023_2024.csv")
df_dias = pd.read_csv('dia_semana.csv')

In [149]:
df_dias.rename(columns={'data':'sale_date'}, inplace=True)

In [150]:
df_final = pd.merge(df_dias,df_vendas,on='sale_date', how='left')
df_final.isnull().sum()

sale_date     0
ano           0
mes           0
nome_mes      0
dia_semana    0
id            8
id_client     8
id_product    8
qtd           8
total         8
dtype: int64

In [151]:
df_final = df_final.fillna(0)
df_final.isnull().sum()

sale_date     0
ano           0
mes           0
nome_mes      0
dia_semana    0
id            0
id_client     0
id_product    0
qtd           0
total         0
dtype: int64

In [152]:
df_diario = df_final.groupby(['sale_date', 'id_product'])['qtd'].sum().reset_index()

In [153]:
ultima_semana_2023 = df_diario[
    (df_diario['sale_date'] >= '2023-12-25') & 
    (df_diario['sale_date'] < '2024-01-01')
]

In [154]:
baseline_produtos = ultima_semana_2023.groupby('id_product')['qtd'].mean().reset_index()
baseline_produtos.rename(columns={'qtd':'media_diaria'}, inplace=True)

In [ ]:
periodo_janeiro = pd.date_range(start='2024-01-01', end='2024-01-31')
df_calendario_janeiro = pd.DataFrame({'sale_date':periodo_janeiro})

In [156]:
df_calendario_janeiro['key'] = 1
baseline_produtos['key'] = 1

In [157]:
previsao_diaria = pd.merge(df_calendario_janeiro, baseline_produtos, on='key').drop('key', axis=1)

In [158]:
previsao_diaria['previsao_venda_diaria'] = previsao_diaria['media_diaria'].round(0)

In [159]:
df_produtos = pd.read_csv('concluidos/produtos_raw_tratados.csv')
df_produtos.rename(columns={'code':'id_product'},inplace=True)

In [161]:
df_final = pd.merge(previsao_diaria,df_produtos,on='id_product',how='left')

In [162]:
colunas_finais = ['sale_date', 'id_product', 'name', 'previsao_venda_diaria']
df_relatorio = df_final[colunas_finais]

df_relatorio

,sale_date,id_product,name,previsao_venda_diaria
0,2024-01-01,3.0,Radar Furuno Pulse Leviathan,10.0
1,2024-01-01,7.0,Radar AIS Zen,10.0
2,2024-01-01,10.0,Piloto Automático Simrad Titan Flux Magnum,10.0
3,2024-01-01,11.0,GPS Furuno Swift Leviathan Poseidon,19.0
4,2024-01-01,13.0,Radar Simrad Boost,3.0
...,...,...,...,...
1514,2024-01-31,133.0,Âncora Bruce Core Pulse,9.0
1515,2024-01-31,141.0,Boia de Arqueamento Bruce Nexus Abyss,15.0
1516,2024-01-31,145.0,Boia de Arqueamento Delta Nexus,2.0
1517,2024-01-31,148.0,Âncora Delta Force Barracuda Mako,12.0


In [165]:
venda_semanal_motor155hp = df_relatorio[
    (df_relatorio['id_product'] == 54.0) & 
    (df_relatorio['sale_date'] >= '2024-01-01') & 
    (df_relatorio['sale_date'] <= '2024-01-07')
]

total_previsao = venda_semanal_motor155hp['previsao_venda_diaria'].sum()

print(f"Previsão total para semana de janeiro: {int(total_previsao)}")

Previsão total para semana de janeiro: 0
